# setup

In [1]:
import os
os.environ["JAVA_HOME"] = "C:\\Program Files\\Eclipse Adoptium\\jdk-17.0.18.8-hotspot"  # adjust version number to match
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = "python"

from pyspark import SparkContext, SparkConf

conf = SparkConf().setMaster("local[2]").set("spark.driver.memory", "30g")
sc = SparkContext(conf=conf)
print(sc.version)

KeyboardInterrupt: 

In [ ]:
import multiprocessing
print(multiprocessing.cpu_count())

# 1. read the 2 CSVs using spark

In [ ]:
%%time
import csv
import io

def parse_csv_line(line):
    """Parse a single CSV line, handling quoted fields correctly."""
    return next(csv.reader(io.StringIO(line)))

# Read matches CSV
matches_raw = sc.textFile("battlesStaging_12072020_to_12262020_WL_tagged.csv")

# Extract header and broadcast it
matches_header = matches_raw.first()
matches_header_bc = sc.broadcast(matches_header)

# Parse rows into dicts, skipping the header
matches_rdd = (
    matches_raw
    .filter(lambda line: line != matches_header_bc.value)
    .map(parse_csv_line)
    .map(lambda fields: dict(zip(matches_header_bc.value.split(","), fields)))
)

In [ ]:
%%time
# Read cards CSV
cards_raw = sc.textFile("CardMasterListSeason18_12082020.csv")

cards_header = cards_raw.first()
cards_header_bc = sc.broadcast(cards_header)

# Build a broadcast dict: card_id -> card_name
# The cards CSV uses columns: team.card1.id and team.card1.name
cards_dict = (
    cards_raw
    .filter(lambda line: line != cards_header_bc.value)
    .map(parse_csv_line)
    .map(lambda fields: dict(zip(cards_header_bc.value.split(","), fields)))
    .map(lambda row: (row["team.card1.id"], row["team.card1.name"]))
    .collectAsMap()
)
cards_bc = sc.broadcast(cards_dict)

# 2. replace card IDs with card names using Spark Core RDD (distributed computing)

In [ ]:
%%time
def add_card_names(row):
    """Look up and attach card names for all 16 card ID columns."""
    lookup = cards_bc.value
    for side in ("winner", "loser"):
        for i in range(1, 9):
            card_id = row.get(f"{side}.card{i}.id", "")
            row[f"{side}.card{i}.name"] = lookup.get(card_id, None)
    return row

result_rdd = matches_rdd.map(add_card_names)
result_rdd.cache()

count = result_rdd.count()
print("Done!", count, "rows")

# Show first row (mirroring df_result.show(1))
first = result_rdd.first()
for k, v in first.items():
    print(f"  {k}: {v}")

# 3. count wins and losses for all cards

In [ ]:
%%time
# Emit (card_name, (wins, losses)) for every card slot in every match.
# Winner slots contribute (1, 0); loser slots contribute (0, 1).
def emit_card_pairs(row):
    pairs = []
    for i in range(1, 9):
        winner_card = row.get(f"winner.card{i}.name")
        loser_card  = row.get(f"loser.card{i}.name")
        if winner_card:
            pairs.append((winner_card, (1, 0)))  # 1 win, 0 losses
        if loser_card:
            pairs.append((loser_card,  (0, 1)))  # 0 wins, 1 loss
    return pairs

# flatMap -> reduceByKey to sum (wins, losses) per card
card_stats_rdd = (
    result_rdd
    .flatMap(emit_card_pairs)
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    # Restructure to (card_name, win_count, loss_count)
    .map(lambda kv: (kv[0], kv[1][0], kv[1][1]))
    # Sort by win_count descending
    .sortBy(lambda x: -x[1])
)
card_stats_rdd.cache()

# Show top 10 (mirroring df_card_stats.show(10))
print(f"{'card_name':<25} {'win_count':>10} {'loss_count':>12}")
print("-" * 50)
for card_name, win_count, loss_count in card_stats_rdd.take(10):
    print(f"{card_name:<25} {win_count:>10} {loss_count:>12}")

# 4. add win ratio

In [ ]:
%%time
def add_win_ratio(row):
    card_name, win_count, loss_count = row
    total = win_count + loss_count
    win_ratio = round(win_count / total, 3) if total > 0 else 0.0
    return (card_name, win_count, loss_count, win_ratio)

card_stats_rdd = card_stats_rdd.map(add_win_ratio)
card_stats_rdd.cache()

# Show top 10 (mirroring df_card_stats.show(10))
print(f"{'card_name':<25} {'win_count':>10} {'loss_count':>12} {'win_ratio':>10}")
print("-" * 62)
for card_name, win_count, loss_count, win_ratio in card_stats_rdd.take(10):
    print(f"{card_name:<25} {win_count:>10} {loss_count:>12} {win_ratio:>10}")

# 5. results

In [ ]:
%%time
# Sort by win_ratio descending and print all cards
all_stats = card_stats_rdd.sortBy(lambda x: -x[3]).collect()

print(f"{'card_name':<25} {'win_count':>10} {'loss_count':>12} {'win_ratio':>10}")
print("-" * 62)
for card_name, win_count, loss_count, win_ratio in all_stats:
    print(f"{card_name:<25} {win_count:>10} {loss_count:>12} {win_ratio:>10}")